# BirdVoice AI - Inference Only
## CPU-optimized submission for BirdCLEF 2026

Minimal runtime: CPU-only, single TTA, no training

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from glob import glob
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import librosa
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"Device: CPU (forced)")

## Configuration

In [ ]:
class CFG:
    DATA_DIR = '/kaggle/input/competitions/birdclef-2026'
    OUTPUT_DIR = '/kaggle/working'
    
    # Try multiple possible checkpoint paths
    CHECKPOINT_PATHS = [
        '/kaggle/input/datasets/imagimaniac/version8-output/best_model.pth',
        '/kaggle/input/imagimaniac/version8-output/best_model.pth',
        '/kaggle/input/versions/8/best_model.pth',
    ]
    
    SAMPLE_RATE = 32000
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 50
    FMAX = 14000
    DURATION = 5
    
    IMG_SIZE = 224
    INF_DURATION = 5
    TTA_SHIFTS = 1  # Single window for speed
    
    DEVICE = torch.device('cpu')  # Force CPU

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

# Find checkpoint
CHECKPOINT_PATH = None
for p in CFG.CHECKPOINT_PATHS:
    if os.path.exists(p):
        CHECKPOINT_PATH = p
        break
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Device: {CFG.DEVICE}")

## Load Data

In [ ]:
sample_sub = pd.read_csv(os.path.join(CFG.DATA_DIR, 'sample_submission.csv'))
target_species = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES = len(target_species)

print(f"Submission rows: {len(sample_sub)}")
print(f"Target species: {NUM_CLASSES}")

## Model

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Linear(dim, 1))
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1), w

class BirdSEDModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.efficientnet_b1(weights=None)
        old_conv = base.features[0][0]
        base.features[0][0] = nn.Conv2d(1, old_conv.out_channels, old_conv.kernel_size, old_conv.stride, old_conv.padding, bias=False)
        with torch.no_grad():
            base.features[0][0].weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        self.backbone = base.features
        self.feat_dim = 1280
        self.pool = AttentionPool(self.feat_dim)
        self.head = nn.Sequential(nn.LayerNorm(self.feat_dim), nn.Dropout(0.3), nn.Linear(self.feat_dim, num_classes))
        self.frame_head = nn.Linear(self.feat_dim, num_classes)
    
    def forward(self, x):
        f = self.backbone(x)
        if len(f.shape) == 4:
            f = f.mean(2).permute(0, 2, 1)
        elif len(f.shape) == 2:
            f = f.unsqueeze(1)
        clipwise = self.head(self.pool(f)[0])
        framewise = self.frame_head(f)
        return clipwise, framewise

model = BirdSEDModel(NUM_CLASSES)

if CHECKPOINT_PATH:
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
    model.load_state_dict(ckpt['state'])
    model.eval()
    print(f"Loaded checkpoint: epoch {ckpt.get('epoch', '?')+1}, val_acc: {ckpt.get('val_acc', '?')}")
else:
    print("WARNING: No checkpoint found in any expected path")

model = model.to(CFG.DEVICE)
print(f"Model ready on {CFG.DEVICE}")

## Audio Processing

In [ ]:
def load_audio(file_path, sr=CFG.SAMPLE_RATE, duration=CFG.DURATION, offset=None):
    target_len = sr * duration
    try:
        if offset is not None:
            audio, _ = librosa.load(file_path, sr=sr, offset=max(0, offset), duration=duration, mono=True)
        else:
            audio, _ = librosa.load(file_path, sr=sr, duration=duration, mono=True)
    except:
        return np.zeros(target_len, dtype=np.float32)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    return audio[:target_len].astype(np.float32)

def to_melspec(audio, sr=CFG.SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=CFG.N_MELS, n_fft=CFG.N_FFT,
                                         hop_length=CFG.HOP_LENGTH, fmin=CFG.FMIN, fmax=CFG.FMAX)
    mel = librosa.power_to_db(mel, ref=np.max)
    mn, mx = mel.min(), mel.max()
    return ((mel - mn) / (mx - mn + 1e-6)).astype(np.float32) if mx - mn > 1e-6 else np.zeros_like(mel)

print("Audio functions ready")

## Inference

In [ ]:
@torch.no_grad()
def predict_clip(model, audio_path, end_time, device):
    """Single window inference - no TTA for speed"""
    try:
        start = max(0, end_time - CFG.INF_DURATION)
        audio = load_audio(audio_path, offset=start, random_crop=False)
        
        if np.abs(audio).max() < 1e-6:
            return np.random.dirichlet(np.ones(NUM_CLASSES) * 0.1)
        
        mel = to_melspec(audio)
        t = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, (CFG.IMG_SIZE, CFG.IMG_SIZE), mode='bilinear', align_corners=False).to(device)
        
        out, _ = model(t)
        probs = torch.sigmoid(out).cpu().numpy()[0]
        return probs
    except Exception as e:
        print(f"Prediction error: {e}")
        return np.random.dirichlet(np.ones(NUM_CLASSES) * 0.1)

print("Predict function ready")

## Generate Submission

In [ ]:
test_dir = os.path.join(CFG.DATA_DIR, 'test_soundscapes')
submission = sample_sub.copy()
n_pred, n_miss, n_error = 0, 0, 0

print(f"Test directory: {test_dir}")
print(f"Exists: {os.path.exists(test_dir)}")

# Recursive file search - finds files in subdirectories
all_test_files = []
if os.path.exists(test_dir):
    all_test_files = glob(os.path.join(test_dir, '**', '*'), recursive=True)
    all_test_files = [f for f in all_test_files if os.path.isfile(f) and f.endswith(('.ogg', '.wav', '.mp3', '.flac'))]
    print(f"Test files found: {len(all_test_files)}")
    if all_test_files:
        print(f"  Sample files: {[os.path.basename(f) for f in all_test_files[:3]]}")
else:
    print("WARNING: test_soundscapes not found!")

print(f"\nProcessing {len(submission)} rows...")

for idx in tqdm(range(len(submission)), desc='Predicting'):
    row_id = submission.iloc[idx]['row_id']
    
    # Parse row_id to get sound ID and end time
    parts = row_id.rsplit('_', 1)
    if len(parts) == 2:
        sid, et = parts[0], int(parts[1])
    else:
        sid, et = row_id, 5
    
    # Try multiple ways to find the audio file
    audio_file = None
    
    # 1. Try exact match with each extension
    for ext in ['.ogg', '.wav', '.mp3', '.flac']:
        candidate = os.path.join(test_dir, sid + ext)
        if os.path.exists(candidate):
            audio_file = candidate
            break
    
    # 2. Fallback: substring match (finds files containing sid in name)
    if audio_file is None and all_test_files:
        sid_base = sid.replace('_', '')
        for f in all_test_files:
            bn = os.path.basename(f)
            if sid in bn or bn.replace('_', '').replace('.ogg', '').replace('.wav', '').replace('.mp3', '').replace('.flac', '') == sid_base:
                audio_file = f
                break
    
    # 3. If still not found - use better fallback
    if audio_file is None:
        n_miss += 1
        # Use Dirichlet noise instead of uniform - gives some variation
        probs = np.random.dirichlet(np.ones(NUM_CLASSES) * 0.1)
    else:
        # Try prediction with error handling
        try:
            probs = predict_clip(model, audio_file, et, CFG.DEVICE)
            n_pred += 1
        except Exception as e:
            n_error += 1
            probs = np.random.dirichlet(np.ones(NUM_CLASSES) * 0.1)
    
    submission.iloc[idx, 1:] = np.clip(probs, 0.001, 0.999)

print(f"\nPredicted: {n_pred} | Missing: {n_miss} | Errors: {n_error}")

## Save Submission

In [ ]:
submission = submission.fillna(0.001)
for col in target_species:
    submission[col] = submission[col].clip(0.001, 0.999)

output_path = os.path.join(CFG.OUTPUT_DIR, 'submission.csv')
submission.to_csv(output_path, index=False)

print(f"\nSubmission saved: {submission.shape}")
print(f"Min prob: {submission.iloc[:, 1:].min().min():.4f}")
print(f"Max prob: {submission.iloc[:, 1:].max().max():.4f}")
print(f"Unique probs: {len(submission.iloc[:, 1:].stack().unique())}")

print(f"\nFirst 3 rows:")
print(submission.head(3))

file_size = os.path.getsize(output_path)
print(f"\nFile size: {file_size/1024:.1f} KB")

In [ ]:
print("\n✅ Submission complete!")
print(f"Output: {output_path}")